# 3 . EMA-Cross Variant Study

Systematically answers the exploration questions for the price-vs-EMA strategy by **sweeping**
each variant with the fast engine. Base idea: long above the Nth EMA, short below - with every
variant a parameter of `EmaCross`.

**Method:** to compare *entries* fairly (Q1-Q6) we hold the EXIT fixed (SL/TP from the config) so
only the entry changes. Then we study *exits* (Q7-Q9) on a fixed good entry. Each cell prints the
top settings by **Sharpe**; see the Metrics section at the bottom for how to judge them.


## Setup & data
Intraday short-term -> default **5-minute** gold. Use a modest window while exploring; widen for
the final confirmation. Costs are ON (gold spread) - intraday edges live or die on costs.


In [ ]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import dataclasses, pandas as pd
from quant.data import get_ohlcv
from quant.engine import BacktestConfig
from quant.strategies import EmaCross
from quant import analytics as A
from quant.viz import sweep_heatmap
from experiments.base import Experiment

SYMBOL, SOURCE, MARKET = 'PAXGUSDT', 'binance', 'spot'   # or 'XAUUSD' + SOURCE='dukascopy'
TF          = '5m'
START, END  = '2025-06-01', '2026-05-31'
df = get_ohlcv(SYMBOL, TF, start=START, end=END, source=SOURCE, market=MARKET, tz='UTC')
print(f'{len(df):,} {TF} bars  {df.open_time.min()} -> {df.open_time.max()}')

In [ ]:
# baseline execution model: risk 1% per trade, SL 0.6%, TP 2R, realistic gold spread
base_cfg = BacktestConfig(initial_cash=10_000, fee_bps=0, slippage_bps=1.0, spread=0.20,
                          exit_enabled=True, sl_mode='entry_pct', sl_value=0.6,
                          tp_mode='rr', tp_value=2.0, sizing_mode='risk_pct_equity',
                          sizing_value=1.0, allow_short=True)
SHOW = ['num_trades','total_return_pct','win_rate_pct','profit_factor','sharpe','max_drawdown_pct']

def study(name, desc, strat_space=None, cfg_space=None, metric='sharpe', min_trades=30, valid_fn=None, cfg=None):
    # run one experiment on the loaded df; returns the ranked DataFrame
    exp = Experiment(name=name, description=desc, strategy_cls=EmaCross, base_cfg=cfg or base_cfg,
                     symbol=SYMBOL, tf=TF, start=START, end=END,
                     strategy_space=strat_space or {}, cfg_space=cfg_space or {},
                     metric=metric, direction='max', min_trades=min_trades, valid_fn=valid_fn)
    return exp.run(df=df)

## Q1 . Entry trigger: cross vs close-above vs full-candle-above
Does requiring a *close* above the EMA, or a *whole candle* above it, beat a bare *cross*?


In [ ]:
r = study('q1_entry_mode', 'cross vs close vs full_candle entry',
          strat_space={'entry_mode':['cross','close','full_candle'], 'confirm_n':[1], 'exit_mode':['none']})
r[['entry_mode']+SHOW]

## Q2 . Confirmation candles + colour
How many candles must hold the condition, and must they be the right colour (green for long)?


In [ ]:
r = study('q2_confirm', 'confirmation candle count + colour',
          strat_space={'entry_mode':['close','full_candle'], 'confirm_n':[1,2,3,5],
                       'confirm_color':[False,True], 'exit_mode':['none']})
r[['entry_mode','confirm_n','confirm_color']+SHOW].head(12)

## Q3 . Heikin-Ashi vs regular candles


In [ ]:
r = study('q3_heikin', 'regular vs Heikin-Ashi',
          strat_space={'use_heikin_ashi':[False,True], 'entry_mode':['close','full_candle'],
                       'confirm_n':[2,3], 'exit_mode':['none']})
r[['use_heikin_ashi','entry_mode','confirm_n']+SHOW].head(10)

## Q4 . Best time of day / weekday (and what to avoid)


In [ ]:
base = EmaCross(ema_period=50, entry_mode='full_candle', confirm_n=2, exit_mode='none')
res = base.backtest(df, base_cfg)
print('By session:'); display(A.by_session(res.trades))
bh = A.by_hour(res.trades).sort_values('total_pnl', ascending=False)
print('Best & worst hours:'); display(pd.concat([bh.head(4), bh.tail(4)]))
display(A.by_weekday(res.trades))

In [ ]:
r = study('q4_session', 'best trading session', metric='total_return_pct',
          strat_space={'entry_mode':['full_candle'], 'confirm_n':[2], 'exit_mode':['none'],
                       'session':[None,'london','newyork','tokyo','sydney']})
r[['session']+SHOW]

## Q5 . Best timeframe (1m / 5m / 15m / 1h)
Higher timeframes usually cut cost-drag and chop; find the sweet spot for short-term trades.


In [ ]:
rows=[]
for tf in ['1m','5m','15m','1h']:
    d = get_ohlcv(SYMBOL, tf, start=START, end=END, source=SOURCE, market=MARKET, tz='UTC')
    st = EmaCross(ema_period=50, entry_mode='full_candle', confirm_n=2, exit_mode='none').backtest(d, base_cfg).stats
    rows.append({'tf':tf,'bars':len(d),'trades':st['num_trades'],'return%':round(st['total_return_pct'],1),
                 'sharpe':round(st['sharpe'],2),'maxDD%':round(st['max_drawdown_pct'],1),'PF':round(st['profit_factor'],3)})
pd.DataFrame(rows)

## Q6 . Best EMA period + higher-timeframe bias filter
Only take trades in the direction of the HTF EMA (e.g. only long when price is above the 1h EMA).


In [ ]:
r = study('q6_ema_htf', 'EMA period x HTF bias',
          strat_space={'ema_period':[20,50,100,200], 'entry_mode':['full_candle'], 'confirm_n':[2],
                       'exit_mode':['none'], 'htf':[None,'1h'], 'htf_ema':[50]})
r[['ema_period','htf']+SHOW].head(12)

## Q7 . Stop-loss type
Fixed % vs previous-swing structure stop. (ATR stop: `prepare()` adds `atr_14`; build an ATR
stop-level column and use `sl_mode='ref_col'` against it - an easy extension.)


In [ ]:
entry = {'ema_period':[50], 'entry_mode':['full_candle'], 'confirm_n':[2], 'exit_mode':['none']}
r_pct = study('q7_sl_pct', 'fixed % stop', strat_space=entry,
              cfg_space={'sl_mode':['entry_pct'], 'sl_value':[0.3,0.5,0.75,1.0,1.5], 'tp_value':[2.0]})
display(r_pct[['sl_value']+SHOW])
cfg_ref = dataclasses.replace(base_cfg, sl_mode='ref_col', sl_ref_long_col='swing_last_low',
                              sl_ref_short_col='swing_last_high', sl_buffer_pct=0.05, sl_max_ref_risk_pct=2.0)
r_ref = study('q7_sl_swing', 'previous-swing structure stop', strat_space=entry,
              cfg_space={'tp_value':[1.5,2.0,3.0]}, cfg=cfg_ref)
display(r_ref[['tp_value']+SHOW])

## Q8 . Take-profit type (RR multiple vs fixed %)


In [ ]:
r = study('q8_tp_rr', 'RR take-profit', strat_space=entry,
          cfg_space={'sl_value':[0.6], 'tp_mode':['rr'], 'tp_value':[1.0,1.5,2.0,3.0,4.0]})
display(r[['tp_value']+SHOW])
r2 = study('q8_tp_pct', 'fixed-% take-profit', strat_space=entry,
           cfg_space={'sl_value':[0.6], 'tp_mode':['entry_pct'], 'tp_value':[0.5,1.0,2.0,3.0]})
display(r2[['tp_value']+SHOW])

## Q9 . Let winners run - trailing stop vs signal-based ride
Trailing stop ratchets on the high-water mark. Signal rides: `exit_mode='ha_flip'` (exit on first
opposite Heikin-Ashi) or `'below_ema'` (exit when a full candle forms beyond a shorter EMA).


In [ ]:
cfg_trail = dataclasses.replace(base_cfg, tp_mode='none', trail_mode='pct')
r = study('q9_trailing', 'trailing-stop widths', strat_space=entry,
          cfg_space={'sl_value':[0.6], 'trail_value':[0.3,0.5,0.8,1.2,2.0]}, cfg=cfg_trail)
display(r[['trail_value']+SHOW])
r2 = study('q9_signal_exits', 'ride with HA-flip / shorter-EMA exit',
           strat_space={'ema_period':[100], 'entry_mode':['full_candle'], 'confirm_n':[1,2],
                        'exit_mode':['ha_flip','below_ema'], 'exit_ema':[20,50]},
           cfg_space={'sl_value':[0.6]})
display(r2[['exit_mode','exit_ema','confirm_n']+SHOW])

## Metrics - what to focus on
- **Sharpe / Sortino** - risk-adjusted return; primary ranking. Sortino penalises only downside.
- **Profit factor** (gross win / gross loss) - >1.3 decent, >1.5 good, *after* costs.
- **Max drawdown %** - survivability; big return + 90% DD is untradeable.
- **Win rate + payoff** - low win-rate can still win with high payoff; check `expectancy_per_trade`.
- **num_trades / exposure** - enough trades for significance (the `min_trades` filter).
- **Attribution (Q4)** - an edge concentrated in one session is more believable than a diffuse one.

**Judging a winner:** prefer a *plateau* of similar-scoring neighbours (robust), not a lone spike
(overfit); then **validate out-of-sample** and with realistic spread. Fewer tuned knobs = safer.

> Reality check: raw EMA-cross variants often *lose* on 1m gold because spread eats every trade -
> that's a finding. Watch whether confirmation, full-candle entries, an HTF bias, and a higher
> timeframe push Sharpe positive. If nothing clears costs, 'no edge here' is a valuable cheap answer.
